# Matching scRNA cells to scDNA clones with clonealign

clonealign assigns each cell of a single-cell RNA sequencing (scRNA) data set to one of the clones found by single-cell DNA sequencing (scDNA) of the same tumor (Campbell et al., 2019).
It assumes that the expression of a gene in a cell is proportional to the copy number of the gene in the clone of the cell.
In this tutorial we grow a tumor, sequence its cells with scDNA and scRNA, find the clones of the scDNA, assign the scRNA cells to them with clonealign and score the assignments against the answer key.
clonealign runs from its own conda environment, `iscc-clonealign`, which contains R, TensorFlow and the clonealign package.
The notebook runs in about ten minutes on a laptop.

In [1]:
%matplotlib inline
import os, tempfile, warnings
warnings.filterwarnings("ignore", message="max_cells")   # the dissociated sample is a subsample by design

import numpy as np
import pandas as pd
import yaml
from scipy.cluster.hierarchy import fcluster, linkage

from iscc.tumor import GenotypeTumor
from iscc.sample import Resection
from iscc.data import scDNA, scRNA
from iscc.data.human import arm_genome
from iscc.integrations import biological_types

CONFIG = os.path.abspath("tutorial_config.yaml")
os.chdir(tempfile.mkdtemp())        # the data and the outputs of the tool go to a scratch directory

## Grow a tumor

We grow the tutorial configuration to 25,000 cancer cells, a size at which the tumor carries subclones with different copy number profiles.
[Shape the tumor](02_shape_the_tumor.ipynb) describes the configuration.
As in the other tool tutorials, the genome that [`arm_genome`](https://pedrofale.github.io/iscc/reference/arm_genome/) builds has one segment per chromosome arm, and the epithelium is the cell of origin, so that cancer cells share the expression baseline of normal epithelial cells.
The tutorial configuration leaves the copy number dosage of the expression layer of `iscc` off, and here we switch on its per-gene dosage with the values that the `iscc` paper fitted to the dosage of TCGA breast tumors.
Under this law, the expression of a gene at total copy number C is its baseline times 1 + s (C/2 - 1), where the sensitivity s of each gene is drawn from a normal distribution with mean 0.86 and standard deviation 0.61 and clipped to the interval from 0 to 1.
A gene with a sensitivity of 1 follows proportional dosage, the assumption of clonealign, and the expression of a gene with a sensitivity of 0 does not change with its copy number.
The expression layer is a readout, so the tumor grows as it would without it.

In [2]:
config = yaml.safe_load(open(CONFIG))
genome = arm_genome(total_genes=6000, driver_scale=0.4)
config["genome_params"] = genome["genome_params"]
config["selection_params"].update(genome["selection_params"])
with open("config.yaml", "w") as f:
    yaml.safe_dump(config, f)

dosage = {"dosage_law": "total", "dosage_sensitivity_mean": 0.86, "dosage_sensitivity_sd": 0.61}
tumor = GenotypeTumor(config="config.yaml", seed=3,
                      expression_params={"cell_of_origin": "epithelial", "dosage_params": dosage})
tumor.grow(n_steps=3000, target_cancer=25_000, materialize=False)
print(f"{tumor.get_cancer_size():,} cancer cells of {tumor.get_tumor_size():,} cells after {tumor.time:.0f} generations")

sensitivity = tumor.programs.dosage_sensitivity
print(f"genes with proportional dosage: {np.mean(sensitivity == 1):.0%}, "
      f"genes whose expression ignores copy number: {np.mean(sensitivity == 0):.0%}")

25,021 cancer cells of 111,394 cells after 432 generations
genes with proportional dosage: 41%, genes whose expression ignores copy number: 7%


## Sequence the cells with scDNA and scRNA

As in [the HMMcopy tutorial](tool_hmmcopy.ipynb), we dissociate the specimen, enrich the suspension for epithelial cells by keeping 5% of the stromal and immune cells, and sequence 400 cells with the DLP+ setting of the [`scDNA`](https://pedrofale.github.io/iscc/reference/scDNA/) assay.
We measure the cancer cells of the same suspension with the 10x setting of the [`scRNA`](https://pedrofale.github.io/iscc/reference/scRNA/) assay.
A study keeps the cancer cells after annotating the cell types, and here we take the cell types from the answer key.

In [3]:
specimen = Resection(tumor)
cells = specimen.dissociate(max_cells=6000, layers=["scdna", "scrna"],
                            recovery_probs={"stromal": 0.05, "immune": 0.05}, seed=0)
cell_type = pd.Series(biological_types(cells["cell_type"]["cell_id"]), index=cells["cell_type"].index)

dlp = dict(breadth="wgs", mu_depth=830, kappa=2.0e5, kappa_loci=5787, ado_rate=0.05)
dna = scDNA(n_cells=400, doublet_rate=0.0, seed=1, **dlp).run(cells)
rna = scRNA(protocol="10x", seed=2).run(cells, cell_subset=cell_type.index[cell_type == "cancer"])
counts = rna.observed_counts                               # cells x genes
print("scDNA cells:", cell_type[dna.cells].value_counts().to_dict())
print(f"scRNA: {counts.shape[0]:,} cancer cells, {counts.shape[1]:,} genes, "
      f"{counts.sum(axis=1).median():,.0f} UMIs per cell (median)")

scDNA cells: {'cancer': 276, 'epithelial': 66, 'stromal': 58}
scRNA: 1,249 cancer cells, 6,001 genes, 2,778 UMIs per cell (median)


## Find the clones of the scDNA

A study finds the clones of a scDNA run by calling copy number in every cell, for example with [HMMcopy](tool_hmmcopy.ipynb), and clustering the cells by their copy number profiles.
At DLP+ depth a shortcut is enough here.
We average the read count of each cell over each segment of the genome, which is a chromosome arm, divide it by the mean of the cell and multiply it by two, which gives the copy number of each arm relative to a diploid genome.
We round these values and cluster the cells by average linkage on the sum of the absolute differences between their profiles, cutting the tree at a distance of 4.5 copies.
We keep the clusters of at least 10 cells, except the cluster whose median profile is flat, which contains the normal cells.
clonealign normalizes the expected expression of a clone over the genes, so it needs copy number only up to a factor per clone, and we give it the mean relative copy number of each clone without rounding.

In [4]:
segment = dna.to_anndata().var["segment"]                  # the segment of every gene
depth = dna.coverage.T.groupby(segment.values).mean().T    # cells x segments
relative = 2 * depth.div(depth.mean(axis=1), axis=0)
tree = linkage(relative.round().values, method="average", metric="cityblock")
cluster = pd.Series(fcluster(tree, t=4.5, criterion="distance"), index=relative.index)
size = cluster.value_counts()
flat = relative.groupby(cluster).median().round().eq(2).all(axis=1)
found = size.index[(size >= 10) & ~flat[size.index].to_numpy()]
clones = relative.groupby(cluster).mean().loc[found]
clones.index = [f"dna_{i}" for i in range(1, len(found) + 1)]
clones.round(1).assign(cells=size[found].to_numpy())

,0,1,2,3,4,5,6,7,8,9,...,30,31,32,33,34,35,36,37,38,cells
dna_1,2.0,2.1,2.1,3.1,1.0,4.2,2.1,1.0,2.1,2.1,...,2.1,2.1,2.1,2.1,2.1,2.0,2.0,2.0,2.1,164
dna_2,1.0,2.3,2.1,3.0,0.7,3.2,2.0,0.9,2.0,2.2,...,2.1,2.1,2.1,2.1,2.1,2.0,2.2,1.9,2.0,21
dna_3,0.0,1.9,2.0,3.0,1.0,3.1,2.0,0.8,2.0,3.0,...,2.0,2.2,2.0,2.0,2.0,1.9,1.9,3.0,2.0,17
dna_4,2.0,2.6,2.0,3.1,0.8,4.0,2.0,1.0,1.8,2.1,...,2.0,2.0,2.0,2.0,2.0,1.9,2.0,1.9,2.0,16
dna_5,2.0,2.1,1.0,3.0,0.0,5.8,2.0,1.0,2.0,2.0,...,2.1,2.0,2.0,2.0,2.0,1.9,1.9,2.0,2.0,11
dna_6,2.1,2.1,2.1,3.1,1.1,3.2,2.1,1.1,1.9,2.1,...,2.1,2.1,2.1,2.2,2.2,2.1,2.1,2.0,2.1,11


We found six clones of at least 10 cells among the cancer cells of the scDNA.
They share most of their copy number changes and differ from one another on a few arms.

## The answer key

The answer key holds the true copy number of every scRNA cell.
We put it on the same relative scale and take, for each cell, the scDNA clone whose profile is closest to its own, by the sum of the absolute differences over the arms that clonealign reads.
The likelihood of clonealign needs at least one copy in every clone, so it reads only the arms on which no clone has lost every copy.
The closest clone is the one a perfect assignment would choose.
Not every cell has the profile of a clone found among 400 scDNA cells, and the distance to the closest clone measures how far each cell is from it.

In [5]:
true_cn = cells["cell_cnv"].loc[counts.index]
true_cn = true_cn.T.groupby(segment[true_cn.columns].values).first().T
true_relative = 2 * true_cn.div(true_cn.mean(axis=1), axis=0)
kept = clones.columns[(clones > 0.5).all()]                # the arms on which every clone has a copy
distance = pd.DataFrame(np.abs(true_relative[kept].values[:, None, :] - clones[kept].values[None]).sum(axis=2),
                        index=true_relative.index, columns=clones.index)
closest = distance.idxmin(axis=1)
print(f"distinct true profiles among the scRNA cells: {len(true_relative.round(2).drop_duplicates())}")
pd.DataFrame({"cells": closest.value_counts(),
              "median distance": distance.min(axis=1).groupby(closest).median().round(2)})

distinct true profiles among the scRNA cells: 283


,cells,median distance
dna_1,753,0.72
dna_2,107,3.44
dna_3,82,2.84
dna_4,197,4.01
dna_5,57,2.45
dna_6,53,3.29


The 1,249 scRNA cells have 283 distinct copy number profiles, far more than the six clones of the scDNA.
The cells closest to dna_1 match its profile, with a median distance of 0.72 copies summed over the arms.
The cells closest to the other clones differ from them by about 2.5 to 4 copies, which indicates that their profiles are not among those of the six clones.

## Write the inputs of clonealign

clonealign reads a cells by genes count matrix and a genes by clones copy number matrix.
We write the counts with the writer of the assay and give every gene on the arms that clonealign reads the copy number of its arm in each clone.

In [6]:
rna.write("clonealign", write_h5ad=False)                  # clonealign/umis.csv, cells x genes
gene_segment = segment[counts.columns]
genes = gene_segment.index[gene_segment.isin(kept)]
copy_number = clones.T.loc[gene_segment[genes]].set_axis(genes)   # genes x clones
copy_number.to_csv("clonealign/copy_number.csv")
print(f"{len(genes):,} genes on {len(kept)} of {clones.shape[1]} arms")

5,183 genes on 34 of 39 arms


## Run clonealign

We run clonealign as its documentation does, filtering the genes and cells with `preprocess_for_clonealign` and fitting with `run_clonealign`, which restarts the variational inference from several initializations, all at their defaults.
A cell whose largest assignment probability is below 0.95 stays unassigned.

In [7]:
%%writefile clonealign/clonealign.R
# clonealign (Campbell et al. 2019) at its defaults
# the Python with TensorFlow of this environment, quiet TensorFlow logs, no terminal control codes
Sys.setenv(RETICULATE_PYTHON = file.path(dirname(dirname(R.home())), "bin", "python"),
           TF_CPP_MIN_LOG_LEVEL = "3", TERM = "dumb")
suppressPackageStartupMessages(library(clonealign))
Y <- as.matrix(read.csv("clonealign/umis.csv", row.names = 1, check.names = FALSE))
L <- as.matrix(read.csv("clonealign/copy_number.csv", row.names = 1, check.names = FALSE))
pp <- preprocess_for_clonealign(Y[, rownames(L)], L)
cat(sprintf("preprocess_for_clonealign kept %d cells and %d genes\n",
            length(pp$retained_cells), length(pp$retained_genes)))

tensorflow::set_random_seed(1)
warned <- character()
fit <- withCallingHandlers(
  run_clonealign(pp$gene_expression_data, pp$copy_number_data, print_elbos = FALSE, verbose = FALSE),
  warning = function(w) {
    warned <<- union(warned, trimws(conditionMessage(w)))
    invokeRestart("muffleWarning")
  })
if (length(warned)) cat("clonealign warned:", warned, sep = "\n")
write.csv(data.frame(cell = pp$retained_cells, clone = fit$clone), "clonealign/clones.csv", row.names = FALSE)

Writing clonealign/clonealign.R


In [8]:
!~/miniconda3/envs/iscc-clonealign/bin/Rscript clonealign/clonealign.R

preprocess_for_clonealign kept 1249 cells and 1660 genes


## Score the assignments

We count a cell as correctly assigned when clonealign assigns it to its closest scDNA clone.
Assigning every cell to the clone that is closest for most cells gives the baseline.

In [9]:
assigned = pd.read_csv("clonealign/clones.csv", index_col="cell")["clone"].reindex(closest.index).fillna("not kept")
print(f"cells assigned to a clone: {assigned.isin(clones.index).mean():.0%}")
print(f"cells assigned to their closest clone: {(assigned == closest).mean():.0%}")
print(f"baseline, every cell to {closest.value_counts().index[0]}: {closest.value_counts(normalize=True).iloc[0]:.0%}")
pd.crosstab(closest.rename("closest clone"), assigned.rename("clonealign"))

cells assigned to a clone: 74%
cells assigned to their closest clone: 28%
baseline, every cell to dna_1: 60%


clonealign,dna_1,dna_2,dna_3,dna_4,dna_5,dna_6,unassigned
closest clone,,,,,,,
dna_1,83,138,13,15,0,266,238
dna_2,0,80,3,0,0,7,17
dna_3,0,4,53,0,0,14,11
dna_4,0,23,6,86,0,42,40
dna_5,3,2,0,24,13,5,10
dna_6,0,9,3,1,0,29,11


clonealign assigned 74% of the cells to a clone, and 28% of all cells to their closest clone, below the 60% of assigning every cell to dna_1.
It assigned most cells closest to dna_1, whose profile they match, to dna_6 or dna_2, or left them unassigned.
In contrast, it assigned most cells closest to dna_2 and dna_3 to those clones.
clonealign assumes that expression is proportional to copy number, while here 59% of the genes are partly or fully buffered, and the clones differ on a few arms only, so the expression differences between clones may be smaller than the model expects.
The `iscc` paper reports the same pattern for an assignment under proportional dosage, which was correct for fewer cells on the expression of `iscc` than on expression generated under proportional dosage, and least so in tumors with five or six copy number profiles.

## Make the benchmark harder or easier

The settings of the tumor and of the assays change how hard this benchmark is, and [Shape the tumor](02_shape_the_tumor.ipynb) and [Shape the data](03_shape_the_data.ipynb) show their effects.

- **Dosage.** A `dosage_sensitivity_mean` of 1 with a `dosage_sensitivity_sd` of 0 gives every gene proportional dosage, the assumption of clonealign, and lower values buffer more genes. The gene programs (`program_params`) and the microenvironment (`microenv_params`) of the expression layer move expression further away from copy number.
- **Clones.** The copy number rate of the cancer cells (`cnv_prob` in `cell_params`) sets how many clones differ in copy number and on how many arms.
- **scRNA.** The depth (`mu_lib`), the overdispersion (`dispersion`), the ambient RNA (`ambient_frac`), the doublets (`doublet_rate`) and the dropout (`dropout_mid`) of the assay set the noise around the dosage signal.
- **scDNA.** Fewer cells or a shallower depth find fewer clones, a different cut of the tree finds more or fewer clones, and clones called with [HMMcopy](tool_hmmcopy.ipynb) carry the errors of the caller into the copy number that clonealign reads.

The [`scRNA`](https://pedrofale.github.io/iscc/reference/scRNA/), [`scDNA`](https://pedrofale.github.io/iscc/reference/scDNA/) and [`GenotypeTumor`](https://pedrofale.github.io/iscc/reference/GenotypeTumor/) pages document every parameter.